# US-004：预处理 — 滤波、降采样、重参考、坏导插值

**目标：** 掌握 EEG 预处理的标准操作，理解每步的物理意义和参数选择。

## 0. 准备数据

In [1]:
import mne
import numpy as np
import os
# 目录配置
data_dir = os.path.abspath("./../datasets")
os.makedirs(data_dir, exist_ok=True)
# 每次运行强制更新 config
mne.set_config("MNE_DATA", data_dir)
mne.set_config("MNE_DATASETS_SAMPLE_PATH", data_dir)
print("=" * 80)
print("config file =", mne.get_config_path())
print("=" * 80)
# 打印config 内容,友好格式
config = mne.get_config()
for key, value in config.items():
    print(f"* {key:25s}: {value}")

# 加载 sample 数据，只取 EEG 通道
sample_dir = mne.datasets.sample.data_path()
raw_fname = sample_dir / "MEG" / "sample" / "sample_audvis_raw.fif"
raw: mne.io.RawArray = mne.io.read_raw_fif(raw_fname, preload=True)
raw.pick_types(eeg=True)  # 只保留 EEG 通道
print(f"原始: {raw}")

config file = /Users/usst_ziyi/.mne/mne-python.json
* MNE_BROWSE_RAW_SIZE      : 14.277777777777779,11.597222222222221
* MNE_DATA                 : /Users/usst_ziyi/Programs/QwenPaw/projects/EEG/mne-eeg-learning/datasets
* MNE_DATASETS_SAMPLE_PATH : /Users/usst_ziyi/Programs/QwenPaw/projects/EEG/mne-eeg-learning/datasets
* MNE_PATH                 : ./datasets
Opening raw data file /Users/usst_ziyi/Programs/QwenPaw/projects/EEG/mne-eeg-learning/datasets/MNE-sample-data/MEG/sample/sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Reading 0 ... 166799  =      0.000 ...   277.714 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
原始: <Raw | sample_audvis_raw.fif, 59 x 166800 (277.7 s), ~78.0 MiB, data loaded>


## 1. 滤波

### 1.1 高通滤波 — 去除慢漂移
截止频率通常选 0.1~1 Hz，去除皮肤电位漂移等低频噪声。

In [2]:
raw_highpass = raw.copy().filter(l_freq=1.0, h_freq=None)
print(f"高通滤波后 info: highpass={raw_highpass.info['highpass']} Hz")

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 1983 samples (3.302 s)

高通滤波后 info: highpass=1.0 Hz


### 1.2 低通滤波 — 去除高频噪声
截止频率通常 30~40 Hz（保留到 gamma 可设 80-100 Hz）。

In [3]:
raw_lowpass = raw.copy().filter(l_freq=None, h_freq=40.0)
print(f"低通滤波后 info: lowpass={raw_lowpass.info['lowpass']} Hz")

Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 199 samples (0.331 s)

低通滤波后 info: lowpass=40.0 Hz


### 1.3 带通滤波 — 最常用
一次同时做高通和低通。

In [4]:
raw_bandpass = raw.copy().filter(l_freq=1.0, h_freq=40.0)
print(f"带通 1-40 Hz: {raw_bandpass.info['highpass']}-{raw_bandpass.info['lowpass']} Hz")


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1983 samples (3.302 s)



带通 1-40 Hz: 1.0-40.0 Hz


### 1.4 陷波滤波 — 去除工频干扰
中国/欧洲 50 Hz，北美 60 Hz。

In [5]:
# 注意：sample 数据采样率 600Hz，Nyquist=300Hz，50Hz 在范围内
raw_notch = raw.copy().notch_filter(freqs=50.0)
print("50 Hz 陷波完成")
print(f"电力线频率: {raw.info['line_freq']} Hz")
# 查看滤波历史
print(raw_notch.info['proc_history'])

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 3965 samples (6.602 s)

50 Hz 陷波完成
电力线频率: None Hz
[]


### 1.5 滤波参数详解

| 参数 | 含义 | 典型值 |
|------|------|--------|
| `l_freq` / `h_freq` | 截止频率 | 1 Hz / 40 Hz |
| `l_trans_bandwidth` | 低端过渡带宽 | 'auto' 或 0.5 Hz |
| `h_trans_bandwidth` | 高端过渡带宽 | 'auto' 或 10 Hz |
| `fir_design`* | FIR 滤波器设计方法* | 'firwin'（默认） |
| `filter_length` | 滤波器长度 | 'auto'（自动计算） |
| `phase` | 相位 | 'zero'（零相位，默认）|

**关键：** MNE 默认使用零相位 FIR 滤波，不会引入相位延迟。

In [ ]:
%matplotlib qt
# 对比滤波前后的频谱
before = raw.compute_psd(fmax=80).plot()          # 滤波前
after = raw_bandpass.compute_psd(fmax=80).plot()  # 滤波后


## 2. 降采样

降低采样率以减少计算量和存储。注意：降采样前必须先低通滤波防止混叠！

In [6]:
print(f"原始采样率: {raw_bandpass.info['sfreq']} Hz")

# 降采样到 150 Hz（注意先做了 40 Hz 低通，满足 Nyquist）
raw_resampled = raw_bandpass.copy().resample(sfreq=150)
print(f"降采样后采样率: {raw_resampled.info['sfreq']} Hz")
print(f"数据点: {raw.n_times} → {raw_resampled.n_times}")

原始采样率: 600.614990234375 Hz
降采样后采样率: 150.0 Hz
数据点: 166800 → 41657


## 3. 重参考

EEG 测量的是电位差，必须选择参考。常见方式：
- **平均参考**：所有通道平均作为参考
- **乳突参考**：M1/M2 或耳后电极
- **单通道参考**：Cz、FCz 等

In [7]:
# 方式一：平均参考
raw_avg_ref = raw_resampled.copy().set_eeg_reference(ref_channels='average')
print("已设为平均参考")

# 方式二：指定通道为参考（如双侧乳突）
# raw_mastoid_ref = raw_resampled.copy().set_eeg_reference(ref_channels=['M1', 'M2'])

# 方式三：REST 参考（无穷远参考，需要安装 mne_connectivity 或特定流程）
# REST 是更现代的方法，但需要额外的 BEM 模型

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
已设为平均参考


## 4. 坏导检测与插值

坏导 = 信号质量差的通道。MNE 可以手动标记或自动检测，然后用邻居插值。

In [8]:
# 4.1 手动标记坏导
# 交互模式下可以在 raw.plot() 中点选
# 这里手动演示
raw_avg_ref.info['bads'] = []  # 清空已有坏导
print(f"当前坏导: {raw_avg_ref.info['bads']}")


当前坏导: []


In [ ]:

# 假设某些通道坏了（随机选一个演示）
import random
fake_bad = random.choice(raw_avg_ref.ch_names)
raw_avg_ref.info['bads'] = [fake_bad]
print(f"假设坏导: {raw_avg_ref.info['bads']}")

# 插值坏导
# 基于球面样条插值，用周围好通道的信号重建坏导
raw_avg_ref.info['bads'] = [fake_bad]  # 恢复我们的假坏导
raw_interp = raw_avg_ref.copy().interpolate_bads(reset_bads=True) # 插值坏导
print(f"插值后坏导列表: {raw_interp.info['bads']}")  # 应为空

假设坏导: ['EEG 024']
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 91.2 mm
Computing interpolation matrix from 58 sensor positions
Interpolating 1 sensors
插值后坏导列表: []


In [18]:
# 基于方差检测
from scipy import stats
variances = raw_avg_ref.get_data(picks='eeg').var(axis=1)
z_vars = stats.zscore(variances)
bads_var = [raw_avg_ref.ch_names[i] for i, z in enumerate(z_vars) if abs(z) > 3]

# 设置坏道
raw_avg_ref.info['bads'] = list(set(bads_var))
print(f"检测到的坏道: {raw_avg_ref.info['bads']}")

检测到的坏道: ['EEG 001', 'EEG 007', 'EEG 003']


In [ ]:
# 补充说明：什么是 z-score
import numpy as np

data = np.array([2, 4, 6, 8, 10])

# 1. 方差（Variance）
variance = data.var()
print(f"方差 = {variance}")  # 8.0

# 2. 标准差（Standard Deviation）
std_dev = data.std()
print(f"标准差 = {std_dev}")  # 2.828

# 3. z 分数（z-score）
z_scores = (data - data.mean()) / data.std()
print(f"z 分数 = {z_scores}")  # [-1.414, -0.707, 0, 0.707, 1.414]

方差 = 8.0
标准差 = 2.8284271247461903
z 分数 = [-1.41421356 -0.70710678  0.          0.70710678  1.41421356]


## 5. 完整预处理 Pipeline

将以上步骤串起来：

In [ ]:
# ===== 一键预处理 =====
def preprocess_pipeline(raw):
    """标准 EEG 预处理流程"""
    # 1. 陷波去工频
    raw = raw.copy().notch_filter(freqs=50)
    # 2. 带通滤波 1-40 Hz
    raw = raw.filter(l_freq=1.0, h_freq=40.0)
    # 3. 降采样
    raw = raw.resample(sfreq=150)
    # 4. 重参考
    raw = raw.set_eeg_reference(ref_channels='average')
    # 5. 插值坏导
    raw.interpolate_bads(reset_bads=True)
    return raw

# 重新加载原始数据测试，不排除坏导，之后会插值坏导
raw_test = mne.io.read_raw_fif(raw_fname, preload=True).pick_types(eeg=True, exclude=[]) 
raw_clean = preprocess_pipeline(raw_test)
print(f"预处理完成: {raw_clean}")
print(f"滤波: {raw_clean.info['highpass']}-{raw_clean.info['lowpass']} Hz")
print(f"采样率: {raw_clean.info['sfreq']} Hz")

Opening raw data file /Users/usst_ziyi/Programs/QwenPaw/projects/EEG/mne-eeg-learning/datasets/MNE-sample-data/MEG/sample/sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Reading 0 ... 166799  =      0.000 ...   277.714 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88

## 6. 练习

1. 尝试不同的高通截止频率（0.1, 0.5, 2 Hz），观察 ERP 波形差异
2. 对比平均参考 vs REST 参考的效果
3. 手动添加一段大振幅方波（模拟坏段），看看滤波对它的影响